© 2026 by Tamás Takács is licensed under CC BY-NC-SA 4.0. To view a copy of this license, visit https://creativecommons.org/licenses/by-nc-sa/4.0/

English translation managed by Tamás Takács. The translation was produced with AI assistance.

# HAIO 2024 - Needle in a Haystack Solution

This notebook contains the solution to the "Needle in a Haystack" problem from the Hungarian Artificial Intelligence Olympiad (HAIO) 2024.

The problem consists of two parts:
1. **PyTorch part**: RNN/CNN-based binary classification on binary sequences
2. **Mistral API part**: LLM "Needle in a Haystack" testing

The problem draws inspiration from Greg Kamradt's "Needle in a Haystack" test.

## Installation and imports

In [ ]:
!pip install mistralai --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import os
import copy
from collections import defaultdict

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Data generation

The `create_dataset` function generates binary sequences. A "needle" pattern (`101010...`) is inserted at a random position into the positive samples, while the negative samples are random binary sequences.

In [ ]:
def create_dataset(n_pos, n_neg, seq_len, needle_len=20):
    """Create a TensorDataset consisting of binary sequences.
    The positive samples contain the needle ('101010...') at a random position.
    The negative samples are random binary sequences."""
    needle = torch.tensor([int(i % 2 == 0) for i in range(needle_len)], dtype=torch.float32)
    data = []
    labels = []
    for _ in range(n_pos):
        seq = torch.randint(0, 2, (seq_len,), dtype=torch.float32)
        pos = torch.randint(0, seq_len - needle_len + 1, (1,)).item()
        seq[pos:pos+needle_len] = needle
        data.append(seq)
        labels.append(1.0)
    for _ in range(n_neg):
        seq = torch.randint(0, 2, (seq_len,), dtype=torch.float32)
        data.append(seq)
        labels.append(0.0)
    data = torch.stack(data).unsqueeze(-1)  # (N, seq_len, 1)
    labels = torch.tensor(labels).unsqueeze(-1)  # (N, 1)
    return TensorDataset(data, labels)

In [ ]:
# Creating the base datasets
SEQ_LEN = 40
NEEDLE_LEN = 20

train_dataset = create_dataset(500, 500, SEQ_LEN, NEEDLE_LEN)
val_dataset = create_dataset(100, 100, SEQ_LEN, NEEDLE_LEN)
test_dataset = create_dataset(200, 200, SEQ_LEN, NEEDLE_LEN)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Training samples: {len(train_dataset)}, Validation: {len(val_dataset)}, Test: {len(test_dataset)}")

## The baseline (faulty) RNN model

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.rnn(x)
        out = self.fc(h_n[-1])
        return out

---
## Task 1: Loss function (5 points)

Let's compute the BCE loss of the untrained model on one batch! Use the `BCEWithLogitsLoss` function.

In [ ]:
# Creating the untrained model
model_untrained = SimpleRNN().to(device)

# Loss function: BCEWithLogitsLoss (works on logits, no sigmoid needed at the end of the model)
criterion = nn.BCEWithLogitsLoss()

# Evaluating one batch
batch_data, batch_labels = next(iter(train_loader))
batch_data, batch_labels = batch_data.to(device), batch_labels.to(device)

with torch.no_grad():
    logits = model_untrained(batch_data)
    loss = criterion(logits, batch_labels)

print(f"Loss of the untrained model: {loss.item():.4f}")
print(f"Expected value (random guessing): {-np.log(0.5):.4f} (= ln(2))")
print(f"\nThe loss of the untrained model is close to ln(2) ≈ 0.693,")
print(f"which indicates that the model is guessing randomly.")

---
## Task 2: False negative probability (10 points)

What is the probability that a random binary sequence (of length `l`) contains the needle pattern of length `k`?

### Formula

The needle pattern (`101010...`) is a specific `k`-bit sequence. At a given position, the probability that the needle appears is $(1/2)^k$.

In a sequence of length `l` there are `l - k + 1` possible starting positions. The probability that the needle does **not** appear at any given position is:

$$P(\text{nincs tű egy pozíción}) = 1 - (1/2)^k$$

If we treat the positions as approximately independent (which is a good approximation for large `l` and a small `k/l` ratio):

$$P(\text{nincs tű sehol}) \approx \left(1 - (1/2)^k\right)^{l-k+1}$$

$$P(\text{van tű}) \approx 1 - \left(1 - 2^{-k}\right)^{l-k+1}$$

**Note**: This is an approximation, because neighbouring positions are not entirely independent. But for large `k` it is very accurate.

This is also the false negative rate: the probability that a sample labelled as negative actually contains the needle.

In [ ]:
def false_negative_probability(seq_len, needle_len=20):
    """The probability that a random binary sequence contains the needle.
    This is also the false negative rate among the negative samples."""
    n_positions = seq_len - needle_len + 1
    p_match_at_pos = 2 ** (-needle_len)  # The probability that the needle appears at one position
    p_no_match_anywhere = (1 - p_match_at_pos) ** n_positions
    p_contains_needle = 1 - p_no_match_anywhere
    return p_contains_needle


# For various sequence lengths
print("Sequence length | P(contains the needle) | False negative rate")
print("-" * 60)
for sl in [40, 80, 160, 320, 640, 1000, 10000, 100000, 1000000]:
    p = false_negative_probability(sl, NEEDLE_LEN)
    print(f"{sl:>12} | {p:>20.10e} | {p*100:.8f}%")

print(f"\nFor k=20, the probability of the needle appearing at one position: {2**(-20):.2e}")
print(f"So in a sequence of length 40 (21 positions): {false_negative_probability(40, 20):.2e}")
print(f"This is extremely small, our negative samples practically never contain the needle.")

---
## Task 3: Improve the training code (5 points)

The problems with the baseline training:
- The SGD optimizer converges poorly for RNNs
- Not enough epochs
- No appropriate learning rate

**Fix**: Adam optimizer, learning rate of 0.001, enough epochs.

In [ ]:
def evaluate_model(model, data_loader, device):
    """Evaluating the model: computing accuracy and loss."""
    model.eval()
    criterion = nn.BCEWithLogitsLoss()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for data, labels in data_loader:
            data, labels = data.to(device), labels.to(device)
            outputs = model(data)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * data.size(0)
            predictions = (torch.sigmoid(outputs) > 0.5).float()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total, total_loss / total


def train_model(model, train_loader, val_loader, device, epochs=30, lr=0.001):
    """Training the model with the Adam optimizer."""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)  # Adam instead of SGD!

    train_losses = []
    val_accs = []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        n_batches = 0
        for data, labels in train_loader:
            data, labels = data.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(data)
            loss = criterion(outputs, labels)
            loss.backward()
            # Gradient clipping for stability
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        avg_loss = epoch_loss / n_batches
        train_losses.append(avg_loss)

        val_acc, val_loss = evaluate_model(model, val_loader, device)
        val_accs.append(val_acc)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs}: "
                  f"Training loss = {avg_loss:.4f}, "
                  f"Validation accuracy = {val_acc:.4f}")

    return train_losses, val_accs

In [ ]:
# Training the model with the improved settings
model_rnn = SimpleRNN(hidden_size=64).to(device)

print("=== Improved training (Adam, lr=0.001) ===")
train_losses, val_accs = train_model(model_rnn, train_loader, val_loader, device, epochs=30, lr=0.001)

# Final evaluation
test_acc, test_loss = evaluate_model(model_rnn, test_loader, device)
print(f"\nTest accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

In [ ]:
# Learning curve visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training loss')
ax1.grid(True)

ax2.plot(val_accs)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation accuracy')
ax2.axhline(y=0.9, color='r', linestyle='--', label='90% target')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

---
## Task 4: Longer sequences (5 points)

Let's examine how the model trained on sequences of length 40 performs on longer sequences!

In [ ]:
# Testing on various sequence lengths
seq_lengths = [40, 60, 80, 120, 160, 240, 320]
accuracies = []

for sl in seq_lengths:
    test_ds = create_dataset(200, 200, sl, NEEDLE_LEN)
    test_dl = DataLoader(test_ds, batch_size=64, shuffle=False)
    acc, _ = evaluate_model(model_rnn, test_dl, device)
    accuracies.append(acc)
    print(f"Sequence length: {sl:>4d} -> Accuracy: {acc:.4f}")

# Visualization
plt.figure(figsize=(8, 5))
plt.plot(seq_lengths, accuracies, 'bo-', markersize=8, linewidth=2)
plt.axhline(y=0.5, color='r', linestyle='--', label='Random guessing')
plt.xlabel('Sequence length')
plt.ylabel('Accuracy')
plt.title('SimpleRNN accuracy at various sequence lengths\n(trained at length 40)')
plt.legend()
plt.grid(True)
plt.ylim(0.4, 1.05)
plt.show()

print("\nObservation: The performance of the RNN typically degrades for longer sequences,")
print("because it struggles to learn long-range dependencies.")

---
## Task 5: Other RNN architectures (5 points)

Let's compare the RNN, LSTM and GRU architectures!

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)  # LSTM returns c_n as well as h_n
        out = self.fc(h_n[-1])
        return out


class GRUModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.gru(x)
        out = self.fc(h_n[-1])
        return out

In [ ]:
# Training all three models
models = {
    'RNN': SimpleRNN(hidden_size=64).to(device),
    'LSTM': LSTMModel(hidden_size=64).to(device),
    'GRU': GRUModel(hidden_size=64).to(device),
}

results = {}
for name, model in models.items():
    print(f"\n=== Training {name} ===")
    train_losses, val_accs = train_model(model, train_loader, val_loader, device, epochs=30, lr=0.001)
    test_acc, _ = evaluate_model(model, test_loader, device)
    results[name] = {'val_accs': val_accs, 'test_acc': test_acc, 'model': model}
    print(f"{name} test accuracy: {test_acc:.4f}")

In [ ]:
# Comparing the learning curves
plt.figure(figsize=(8, 5))
for name, res in results.items():
    plt.plot(res['val_accs'], label=f"{name} (test: {res['test_acc']:.3f})")
plt.xlabel('Epoch')
plt.ylabel('Validation accuracy')
plt.title('RNN vs LSTM vs GRU - Validation accuracy')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Comparison on longer sequences
seq_lengths = [40, 60, 80, 120, 160, 240, 320]

plt.figure(figsize=(10, 6))
for name, res in results.items():
    accs = []
    for sl in seq_lengths:
        test_ds = create_dataset(200, 200, sl, NEEDLE_LEN)
        test_dl = DataLoader(test_ds, batch_size=64, shuffle=False)
        acc, _ = evaluate_model(res['model'], test_dl, device)
        accs.append(acc)
    plt.plot(seq_lengths, accs, 'o-', markersize=6, linewidth=2, label=name)

plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
plt.xlabel('Sequence length')
plt.ylabel('Accuracy')
plt.title('Comparison of architectures across different sequence lengths')
plt.legend()
plt.grid(True)
plt.ylim(0.4, 1.05)
plt.show()

print("Observation: LSTM and GRU typically perform better on longer sequences,")
print("because they can learn longer-range dependencies thanks to their gating mechanisms.")

---
## Task 6: Attribution / Gradients (10 points)

Let's examine which positions the model attends to! Gradient-based attribution shows how much each input position influences the output.

In [ ]:
def compute_input_gradients(model, x):
    """Compute input gradients with respect to the model output.
    
    Args:
        model: The trained model
        x: Input tensor (1, seq_len, 1)
    Returns:
        Absolute value of the gradients (seq_len,)
    """
    model.eval()
    x = x.clone().detach().requires_grad_(True)
    output = model(x)
    output.backward()
    # The absolute value of the gradient shows the importance of each position
    gradients = x.grad.squeeze().abs()
    return gradients.detach().cpu().numpy()

In [ ]:
# Generating a positive sample with a known needle position
needle = torch.tensor([int(i % 2 == 0) for i in range(NEEDLE_LEN)], dtype=torch.float32)

# Create a positive sample where we know the needle position
seq_len_test = 80
needle_pos = 30  # The needle starts at position 30
test_seq = torch.randint(0, 2, (seq_len_test,), dtype=torch.float32)
test_seq[needle_pos:needle_pos+NEEDLE_LEN] = needle
test_input = test_seq.unsqueeze(0).unsqueeze(-1).to(device)  # (1, seq_len, 1)

# Let's use the LSTM model (it usually gives better attribution)
best_model = results['LSTM']['model']
grads = compute_input_gradients(best_model, test_input)

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# 1. The input sequence
axes[0].bar(range(seq_len_test), test_seq.numpy(), color='steelblue', alpha=0.7)
axes[0].axvspan(needle_pos, needle_pos + NEEDLE_LEN, alpha=0.2, color='red', label=f'Needle position ({needle_pos}-{needle_pos+NEEDLE_LEN})')
axes[0].set_ylabel('Value')
axes[0].set_title('Input sequence')
axes[0].legend()

# 2. Gradients
colors = ['red' if needle_pos <= i < needle_pos + NEEDLE_LEN else 'steelblue' for i in range(seq_len_test)]
axes[1].bar(range(seq_len_test), grads, color=colors, alpha=0.7)
axes[1].axvspan(needle_pos, needle_pos + NEEDLE_LEN, alpha=0.1, color='red')
axes[1].set_ylabel('|Gradient|')
axes[1].set_title('Input gradients (red = needle position)')

# 3. Heatmap
axes[2].imshow(grads.reshape(1, -1), aspect='auto', cmap='hot', interpolation='nearest')
axes[2].axvline(x=needle_pos, color='cyan', linewidth=2, linestyle='--')
axes[2].axvline(x=needle_pos + NEEDLE_LEN, color='cyan', linewidth=2, linestyle='--')
axes[2].set_xlabel('Position')
axes[2].set_title('Gradient heatmap (cyan lines = needle boundaries)')
axes[2].set_yticks([])

plt.tight_layout()
plt.show()

# Comparison: average gradient at needle vs non-needle positions
needle_grads = grads[needle_pos:needle_pos+NEEDLE_LEN].mean()
non_needle_grads = np.concatenate([grads[:needle_pos], grads[needle_pos+NEEDLE_LEN:]]).mean()
print(f"Average gradient at the needle positions: {needle_grads:.6f}")
print(f"Average gradient at the other positions: {non_needle_grads:.6f}")
print(f"Ratio: {needle_grads/max(non_needle_grads, 1e-10):.2f}x")

In [ ]:
# Averaged over several samples
n_samples = 20
avg_grads = np.zeros(seq_len_test)
actual_needle_pos = needle_pos  # We always place the needle at the same spot

for _ in range(n_samples):
    seq = torch.randint(0, 2, (seq_len_test,), dtype=torch.float32)
    seq[actual_needle_pos:actual_needle_pos+NEEDLE_LEN] = needle
    inp = seq.unsqueeze(0).unsqueeze(-1).to(device)
    g = compute_input_gradients(best_model, inp)
    avg_grads += g

avg_grads /= n_samples

plt.figure(figsize=(12, 4))
colors = ['red' if actual_needle_pos <= i < actual_needle_pos + NEEDLE_LEN else 'steelblue' for i in range(seq_len_test)]
plt.bar(range(seq_len_test), avg_grads, color=colors, alpha=0.7)
plt.axvspan(actual_needle_pos, actual_needle_pos + NEEDLE_LEN, alpha=0.1, color='red', label='Needle position')
plt.xlabel('Position')
plt.ylabel('Average |gradient|')
plt.title(f'Average gradient attribution (over {n_samples} samples)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## Task 7: Arguments for convolution (5 points)

**Why would a CNN be suitable for this task?**

The needle pattern (`101010...`) is a **fixed-length, local pattern** within the input sequence. Convolutional neural networks (CNNs) were designed precisely to recognize such local patterns. A 1D convolutional layer whose kernel size is at least as large as the needle length can detect the presence of the needle with a single convolution operation. The CNN does not have to remember earlier elements of the sequence (unlike an RNN); it simply slides the filter across the input and checks everywhere whether the local pattern matches the needle. Moreover, a CNN generalizes well to longer sequences by its very nature, since it applies the same filter at every position of the sequence, regardless of the total sequence length. This translation invariance is what makes it particularly well suited to the task.

---
## Task 8: CNN implementation (5 points)

A 1D CNN implementation for finding the needle.

In [ ]:
class NeedleCNN(nn.Module):
    """1D convolutional network for detecting the needle.
    
    The Conv1d kernel size is at least the needle length, so it can
    recognize the pattern in a single convolution step.
    """
    def __init__(self, needle_len=20, n_filters=32):
        super().__init__()
        # First convolutional layer: kernel size = needle length
        self.conv1 = nn.Conv1d(1, n_filters, kernel_size=needle_len, padding=0)
        self.relu = nn.ReLU()
        # Second convolutional layer with a smaller kernel
        self.conv2 = nn.Conv1d(n_filters, 16, kernel_size=3, padding=1)
        # Global max pooling + linear layer
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        # x: (batch, seq_len, 1) -> (batch, 1, seq_len) for Conv1d
        x = x.transpose(1, 2)
        x = self.relu(self.conv1(x))   # (batch, n_filters, seq_len - needle_len + 1)
        x = self.relu(self.conv2(x))   # (batch, 16, ...)
        # Global max pooling: takes the maximum along the sequence
        x = x.max(dim=2)[0]            # (batch, 16)
        x = self.fc(x)                 # (batch, 1)
        return x

In [ ]:
# Training the CNN
model_cnn = NeedleCNN(needle_len=NEEDLE_LEN, n_filters=32).to(device)

print("=== Training the CNN ===")
cnn_train_losses, cnn_val_accs = train_model(model_cnn, train_loader, val_loader, device, epochs=30, lr=0.001)

# Test accuracy
cnn_test_acc, cnn_test_loss = evaluate_model(model_cnn, test_loader, device)
print(f"\nCNN test accuracy: {cnn_test_acc:.4f}")

In [ ]:
# Comparison of CNN vs RNN vs LSTM vs GRU across different sequence lengths
seq_lengths = [40, 60, 80, 120, 160, 240, 320]

all_models = {
    'RNN': results['RNN']['model'],
    'LSTM': results['LSTM']['model'],
    'GRU': results['GRU']['model'],
    'CNN': model_cnn,
}

plt.figure(figsize=(10, 6))
for name, model in all_models.items():
    accs = []
    for sl in seq_lengths:
        test_ds = create_dataset(200, 200, sl, NEEDLE_LEN)
        test_dl = DataLoader(test_ds, batch_size=64, shuffle=False)
        acc, _ = evaluate_model(model, test_dl, device)
        accs.append(acc)
    plt.plot(seq_lengths, accs, 'o-', markersize=6, linewidth=2, label=name)

plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
plt.xlabel('Sequence length')
plt.ylabel('Accuracy')
plt.title('Comparison of all architectures across different sequence lengths')
plt.legend()
plt.grid(True)
plt.ylim(0.4, 1.05)
plt.show()

print("Observation: The CNN generalizes well to longer sequences,")
print("since the convolutional filter operates locally and is independent of sequence length.")

---
---
# Mistral API section

In the following tasks we test the Mistral LLM with the "Needle in a Haystack" method.

In [ ]:
# Mistral API initialization
MISTRAL_AVAILABLE = False

try:
    from mistralai import Mistral
    api_key = os.environ.get("MISTRAL_API_KEY", "")
    if api_key:
        client = Mistral(api_key=api_key)
        MISTRAL_AVAILABLE = True
        print("Mistral API successfully initialized.")
    else:
        print("MISTRAL_API_KEY is not set. The API tasks will run with simulated results.")
except Exception as e:
    print(f"Mistral API is not available: {e}")
    print("The API tasks will run with simulated results.")

In [ ]:
def query_mistral(prompt, model_name="mistral-small-latest"):
    """Wrapper for querying the Mistral API."""
    if not MISTRAL_AVAILABLE:
        return None
    try:
        response = client.chat.complete(
            model=model_name,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=256,
            temperature=0.0,
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"API error: {e}")
        return None

---
## Mistral Task 1: Designing the NIAH test (5 points)

### Description of the test

In the "Needle in a Haystack" (NIAH) test we place a specific fact (the "needle") inside a long text (the "haystack"), and then ask the language model about that fact. The goal is to examine whether the model is able to find and return the relevant information from the long context.

**Methodology:**

1. **The needle**: A concrete, unique fact that is unlikely to appear in the model's training data. For example: "The best pizza in Budapest is found at the Tűzrakó Restaurant, whose secret ingredient is truffle olive oil."

2. **The haystack**: English-language text generated from Paul Graham's essays or a similar source. The length of the text varies (e.g. 500, 1000, 2000, 4000, 8000 tokens).

3. **Placing the needle**: We insert the needle at different positions in the text (beginning, 25%, middle, 75%, end) in order to examine whether the position affects retrieval success.

4. **The question**: We ask a question related to the content of the needle, for example: "What is the secret ingredient of the best pizza in Budapest?"

5. **Evaluation**: We check the correctness of the answer (whether it contains the keyword, e.g. "truffle olive oil"). The results are visualized as a heatmap: the x-axis is the needle position, the y-axis is the context length.

---
## Mistral Task 2: NIAH implementation (10 points)

In [ ]:
# The "needle" - a specific fact
NEEDLE_FACT = "The best pizza in Budapest can be found at the Tuzrako Restaurant, whose secret ingredient is truffle olive oil."
NEEDLE_QUESTION = "What is the secret ingredient of the best pizza in Budapest?"
NEEDLE_ANSWER_KEYWORD = "truffle olive oil"

# Haystack text generator (repeating paragraphs)
FILLER_PARAGRAPHS = [
    "Technology has transformed the way we communicate and interact with the world around us. From smartphones to artificial intelligence, innovation continues to reshape every aspect of our daily lives. The pace of change shows no signs of slowing down.",
    "Education plays a crucial role in shaping the future of society. By investing in quality learning experiences, we can empower individuals to reach their full potential. Critical thinking and problem-solving skills are more important than ever.",
    "The natural world offers endless sources of inspiration and wonder. From the depths of the ocean to the peaks of the highest mountains, our planet is home to an incredible diversity of life forms. Conservation efforts are essential to preserve this biodiversity.",
    "Art and culture have been fundamental to human expression throughout history. Music, literature, painting, and dance allow us to explore emotions and ideas that transcend language barriers. Creative expression enriches our understanding of the human experience.",
    "Scientific research continues to push the boundaries of human knowledge. Breakthroughs in medicine, physics, and biology have the potential to solve some of the world's most pressing challenges. Collaboration between disciplines accelerates discovery.",
    "Urban development presents both opportunities and challenges for modern societies. As cities grow, planners must balance economic growth with environmental sustainability. Smart city initiatives leverage technology to improve quality of life for residents.",
    "The global economy is interconnected in ways that previous generations could not have imagined. Trade agreements, digital commerce, and international cooperation drive economic growth across borders. Understanding these dynamics is essential for informed decision-making.",
    "Health and wellness have become central concerns in contemporary culture. People are increasingly aware of the importance of nutrition, exercise, and mental health. Preventive healthcare approaches aim to address issues before they become serious problems.",
]


def generate_haystack(target_word_count):
    """Generate haystack text with the given word count."""
    paragraphs = []
    word_count = 0
    i = 0
    while word_count < target_word_count:
        p = FILLER_PARAGRAPHS[i % len(FILLER_PARAGRAPHS)]
        paragraphs.append(p)
        word_count += len(p.split())
        i += 1
    return paragraphs


def insert_needle(paragraphs, needle, position_ratio):
    """Insert the needle at the given relative position.
    
    Args:
        paragraphs: List of paragraphs
        needle: The fact to insert
        position_ratio: A number between 0.0 (beginning) and 1.0 (end)
    Returns:
        The assembled text containing the needle
    """
    insert_idx = int(len(paragraphs) * position_ratio)
    insert_idx = max(0, min(insert_idx, len(paragraphs)))
    paragraphs_with_needle = paragraphs[:insert_idx] + [needle] + paragraphs[insert_idx:]
    return "\n\n".join(paragraphs_with_needle)


def check_answer(response, keyword):
    """Checks whether the response contains the keyword."""
    if response is None:
        return False
    return keyword.lower() in response.lower()

In [ ]:
def run_niah_test(model_name="mistral-small-latest", word_counts=None, positions=None):
    """Run the NIAH test for various text lengths and needle positions.
    
    Returns:
        results: 2D numpy array (word_counts x positions), 1 = correct, 0 = incorrect
    """
    if word_counts is None:
        word_counts = [200, 500, 1000, 2000, 4000]
    if positions is None:
        positions = [0.0, 0.25, 0.5, 0.75, 1.0]

    results = np.zeros((len(word_counts), len(positions)))

    for i, wc in enumerate(word_counts):
        for j, pos in enumerate(positions):
            paragraphs = generate_haystack(wc)
            full_text = insert_needle(paragraphs, NEEDLE_FACT, pos)
            
            prompt = f"""Read the following text carefully and answer the question at the end.

---
{full_text}
---

Question: {NEEDLE_QUESTION}
Answer concisely:"""

            if MISTRAL_AVAILABLE:
                response = query_mistral(prompt, model_name=model_name)
                success = check_answer(response, NEEDLE_ANSWER_KEYWORD)
                results[i, j] = 1.0 if success else 0.0
                status = "CORRECT" if success else "INCORRECT"
                print(f"  Words: {wc:>5}, Position: {pos:.2f} -> {status}")
            else:
                # Simulated result: better for short texts and extreme positions
                base_prob = max(0.3, 1.0 - wc / 8000)
                pos_penalty = 0.1 * abs(pos - 0.5)  # Harder in the middle
                results[i, j] = 1.0 if np.random.random() < (base_prob - pos_penalty) else 0.0

    return results, word_counts, positions


print("=== Running the NIAH test (natural language) ===")
if not MISTRAL_AVAILABLE:
    print("(Simulated results - Mistral API not available)")

niah_results, word_counts, positions = run_niah_test()
print("\nDone!")

In [ ]:
# NIAH heatmap visualization
def plot_niah_heatmap(results, word_counts, positions, title="NIAH Test Results"):
    """Display the NIAH results as a heatmap."""
    fig, ax = plt.subplots(figsize=(8, 6))
    
    im = ax.imshow(results, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1,
                   interpolation='nearest')
    
    # Axis labels
    pos_labels = [f"{int(p*100)}%" for p in positions]
    ax.set_xticks(range(len(positions)))
    ax.set_xticklabels(pos_labels)
    ax.set_yticks(range(len(word_counts)))
    ax.set_yticklabels([str(wc) for wc in word_counts])
    
    ax.set_xlabel('Position of the needle in the text')
    ax.set_ylabel('Text length (words)')
    ax.set_title(title)
    
    # Write the values into the cells
    for i in range(len(word_counts)):
        for j in range(len(positions)):
            text = "OK" if results[i, j] > 0.5 else "X"
            color = "white" if results[i, j] < 0.5 else "black"
            ax.text(j, i, text, ha="center", va="center", color=color, fontweight='bold')
    
    plt.colorbar(im, label='Success (1) / Failure (0)')
    plt.tight_layout()
    plt.show()


plot_niah_heatmap(niah_results, word_counts, positions,
                  title="NIAH Test - Natural language (Mistral)")

---
## Mistral Task 3: Designing a code-based NIAH test (5 points)

### Description of the code-based NIAH test

In the code-based NIAH test, the "needle" is a specific variable assignment inside a long code snippet. The "haystack" consists of generated Python code containing numerous variable definitions, functions and classes. The needle is a unique, specific assignment that the model has to find.

**Methodology:**

1. **The needle**: A specific variable assignment, for example `secret_answer = 42.7913`. This is a unique value that does not appear anywhere else in the code.

2. **The haystack**: Generated Python code containing random variable definitions, simple functions, lists and dictionaries. The code is syntactically correct but semantically meaningless.

3. **Placing the needle**: We insert the needle at various positions in the code, similarly to the natural-language test.

4. **The question**: "What is the value of the variable `secret_answer`?"

5. **Evaluation**: We check the correctness of the answer based on the exact value.

---
## Mistral Task 4: Implementing the code-based NIAH test (10 points)

In [ ]:
import random
import string

# The code needle
CODE_NEEDLE = "secret_answer = 42.7913"
CODE_QUESTION = "What is the value of the variable `secret_answer` in the code above?"
CODE_ANSWER_KEYWORD = "42.7913"


def generate_random_varname():
    """Generate a random variable name."""
    prefixes = ['data', 'result', 'value', 'count', 'total', 'temp', 'var',
                'num', 'idx', 'flag', 'config', 'param', 'score', 'weight']
    return f"{random.choice(prefixes)}_{random.randint(1, 999)}"


def generate_code_line():
    """Generate a random line of code."""
    templates = [
        lambda: f"{generate_random_varname()} = {random.uniform(-100, 100):.4f}",
        lambda: f"{generate_random_varname()} = {random.randint(-1000, 1000)}",
        lambda: f"{generate_random_varname()} = [{', '.join(str(random.randint(0, 100)) for _ in range(random.randint(3, 7)))}]",
        lambda: f"{generate_random_varname()} = '{random.choice(['alpha', 'beta', 'gamma', 'delta', 'epsilon', 'zeta'])}'",
        lambda: f"# {random.choice(['Compute', 'Calculate', 'Initialize', 'Update', 'Process'])} {generate_random_varname()}",
        lambda: f"{generate_random_varname()} = {generate_random_varname()} + {random.uniform(0, 10):.2f}" if random.random() > 0.5 else f"{generate_random_varname()} = True",
    ]
    return random.choice(templates)()


def generate_code_block(n_lines=10):
    """Generate a code block with n lines."""
    lines = [generate_code_line() for _ in range(n_lines)]
    return "\n".join(lines)


def generate_code_haystack(target_lines):
    """Generate a code haystack with the given number of lines."""
    blocks = []
    current_lines = 0
    while current_lines < target_lines:
        block_size = random.randint(5, 15)
        blocks.append(generate_code_block(block_size))
        current_lines += block_size
    return blocks


def insert_code_needle(blocks, needle, position_ratio):
    """Insert the needle into the code at the given position."""
    insert_idx = int(len(blocks) * position_ratio)
    insert_idx = max(0, min(insert_idx, len(blocks)))
    blocks_with_needle = blocks[:insert_idx] + [needle] + blocks[insert_idx:]
    return "\n\n".join(blocks_with_needle)

In [ ]:
def run_code_niah_test(model_name="mistral-small-latest", line_counts=None, positions=None):
    """Run the code-based NIAH test."""
    if line_counts is None:
        line_counts = [50, 100, 200, 400, 800]
    if positions is None:
        positions = [0.0, 0.25, 0.5, 0.75, 1.0]

    results = np.zeros((len(line_counts), len(positions)))

    for i, lc in enumerate(line_counts):
        for j, pos in enumerate(positions):
            blocks = generate_code_haystack(lc)
            full_code = insert_code_needle(blocks, CODE_NEEDLE, pos)
            
            prompt = f"""Read the following Python code carefully and answer the question.

```python
{full_code}
```

Question: {CODE_QUESTION}
Answer with just the value:"""

            if MISTRAL_AVAILABLE:
                response = query_mistral(prompt, model_name=model_name)
                success = check_answer(response, CODE_ANSWER_KEYWORD)
                results[i, j] = 1.0 if success else 0.0
                status = "CORRECT" if success else "INCORRECT"
                print(f"  Lines: {lc:>5}, Position: {pos:.2f} -> {status}")
            else:
                # Simulated result
                base_prob = max(0.2, 1.0 - lc / 1500)
                pos_penalty = 0.15 * abs(pos - 0.0)  # Easier at the beginning
                results[i, j] = 1.0 if np.random.random() < (base_prob - pos_penalty) else 0.0

    return results, line_counts, positions


print("=== Running the code-based NIAH test ===")
if not MISTRAL_AVAILABLE:
    print("(Simulated results - Mistral API not available)")

code_niah_results, line_counts, positions = run_code_niah_test()
print("\nDone!")

In [ ]:
# Code-based NIAH heatmap
plot_niah_heatmap(code_niah_results, line_counts, positions,
                  title="Code-based NIAH Test Results (Mistral)")

---
## Mistral Task 5: Model comparison (5 points)

Comparing two Mistral models on the NIAH tests: Mistral Small (7B) vs Mistral Large (Mixtral).

In [ ]:
# Model comparison
model_configs = {
    "Mistral Small (7B)": "mistral-small-latest",
    "Mistral Large (Mixtral)": "mistral-large-latest",
}

# Smaller test to reduce costs
comparison_word_counts = [200, 500, 1000, 2000]
comparison_positions = [0.0, 0.25, 0.5, 0.75, 1.0]

comparison_results = {}

for display_name, model_id in model_configs.items():
    print(f"\n=== {display_name} ({model_id}) ===")
    if not MISTRAL_AVAILABLE:
        print("(Simulated results)")
    
    results, wc, pos = run_niah_test(
        model_name=model_id,
        word_counts=comparison_word_counts,
        positions=comparison_positions
    )
    comparison_results[display_name] = results

In [ ]:
# Comparative heatmaps
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (display_name, results) in enumerate(comparison_results.items()):
    ax = axes[idx]
    im = ax.imshow(results, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1,
                   interpolation='nearest')
    
    pos_labels = [f"{int(p*100)}%" for p in comparison_positions]
    ax.set_xticks(range(len(comparison_positions)))
    ax.set_xticklabels(pos_labels)
    ax.set_yticks(range(len(comparison_word_counts)))
    ax.set_yticklabels([str(wc) for wc in comparison_word_counts])
    
    ax.set_xlabel('Needle position')
    ax.set_ylabel('Text length (words)')
    ax.set_title(display_name)
    
    # Values in the cells
    for i in range(len(comparison_word_counts)):
        for j in range(len(comparison_positions)):
            text = "OK" if results[i, j] > 0.5 else "X"
            color = "white" if results[i, j] < 0.5 else "black"
            ax.text(j, i, text, ha="center", va="center", color=color, fontweight='bold')

plt.suptitle('Model Comparison - NIAH Test', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Aggregate scores
print("\n=== Aggregate results ===")
for name, results in comparison_results.items():
    avg_score = results.mean()
    print(f"{name}: {avg_score:.1%} successful retrieval")

---
## Summary

### PyTorch part
- The untrained model's loss is ~ln(2), which corresponds to random guessing
- The false negative rate is extremely low for a 20-bit needle
- The Adam optimizer + suitable hyperparameters significantly improve the RNN's performance
- LSTM and GRU generalize better to longer sequences than a plain RNN
- Gradient attribution can reveal where the needle is in the sequence
- By its very nature, the CNN is well suited to recognizing local patterns and generalizes well

### Mistral API part
- The NIAH test effectively measures context-retrieval capability
- As the text length grows, retrieval becomes harder
- The needle's position also affects the result
- Larger models generally perform better on the NIAH test